# IND320 - Project work, part 2: reservoirs from the NVE API and cross-border power flows

- **Student:** Iftikhar Amiri
- **GitHub repository:** https://github.com/iftikharamiri/ind320 (part 2 is developed on the branch `part2`, then merged into `main`)
- **Streamlit app:** https://ind320-iftikhar-amiri.streamlit.app/
- **Screencast:** SCREENCAST_LINK

## Contents
1. Setup
2. Secrets and database connections (MongoDB, Cassandra)
3. NVE Magasinstatistikk API: areas and time columns
4. Part 1 plots, recreated from the API
5. ENTSO-E: cross-border flows from/to Norway
6. Spark: insert into Cassandra
7. Spark: extract from Cassandra
8. Plots: pie chart and first-month line plot
9. Spark data into MongoDB
10. AI usage
11. Log

## 1. Setup
This notebook must run with the **IND320 Spark (Python 3.11)** kernel: the Python 3.12
kernel crashes PySpark 3.5 on Windows. The environment variables below tell Spark which
Java and which Python to use, and must be set **before** the first Spark session starts.

In [ ]:
import os
from pathlib import Path

# Java 17 (Spark/Cassandra break above 17), winutils and the Python Spark workers use.
# setdefault keeps any value that is already set on the machine.
os.environ.setdefault("JAVA_HOME", r"C:\dev\tools\jdk-17.0.20.1+1")
os.environ.setdefault("HADOOP_HOME", r"C:\Hadoop\hadoop-3.3.1")
os.environ["PYSPARK_PYTHON"] = r"C:\dev\ind320\spark311\Scripts\python.exe"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

import matplotlib.pyplot as plt
import pandas as pd
import requests

# Same colour-blind-checked palette and plot style as in part 1
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
           "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({
    "figure.figsize": (11, 3.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "lines.linewidth": 1.5,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

# Folder with the passwords/tokens. It lives outside the repository on purpose.
SECRETS = Path("../../course-material/No_sync")

## 2. Secrets and database connections
Passwords are read from files in `No_sync/` (never committed to GitHub). In the
Streamlit app the same secrets are given through Streamlit's secrets mechanism.

**MongoDB Atlas** (remote): a free M0 cluster. The notebook writes the curated data there
and the Streamlit app reads it. The password goes through `quote_plus` because special
characters in a password would otherwise break the connection string.

In [ ]:
from urllib.parse import quote_plus
from pymongo import MongoClient

# Line 1 = database user, line 2 = password
mongo_user, mongo_password = (SECRETS / "MongoDB").read_text().split("\n")[:2]
MONGO_HOST = "ind320.w1oh2ln.mongodb.net"
MONGO_URI = f"mongodb+srv://{quote_plus(mongo_user.strip())}:{quote_plus(mongo_password.strip())}@{MONGO_HOST}/"

mongo = MongoClient(MONGO_URI, serverSelectionTimeoutMS=20000)
print("MongoDB ping:", mongo.admin.command("ping"))

Test that data can be manipulated from Python: insert, read, update and delete one
document in a scratch collection.

In [ ]:
test = mongo["ind320"]["connection_test"]
doc_id = test.insert_one({"hello": "mongodb", "n": 1}).inserted_id          # create
print("read   :", test.find_one({"_id": doc_id}))                           # read
test.update_one({"_id": doc_id}, {"$set": {"n": 2}})                        # update
print("updated:", test.find_one({"_id": doc_id}))
test.delete_one({"_id": doc_id})                                            # delete
print("left in collection:", test.count_documents({}))

**Cassandra + Spark** (local): Cassandra runs in the Docker container `my_cassandra`
(`docker start my_cassandra`; it needs about a minute before it accepts connections).
Spark talks to it through the DataStax connector. I use the **assembly** jar of the
connector, because the plain jar makes every Spark job hang with a Netty conflict.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[2]").appName("ind320-part2")
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector-assembly_2.12:3.5.1")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .config("spark.sql.catalog.mycatalog", "com.datastax.spark.connector.datasource.CassandraCatalog")
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.sql.session.timeZone", "UTC")   # all timestamps in UTC
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# A keyspace is Cassandra's "database". SimpleStrategy with factor 1 = a single local node.
spark.sql("""CREATE DATABASE IF NOT EXISTS mycatalog.ind320
             WITH DBPROPERTIES (class='SimpleStrategy', replication_factor='1')""")
spark.sql("SHOW NAMESPACES FROM mycatalog").show()
# A tiny Spark job proves that the Python workers run too
print("Spark job OK:", spark.range(5).count() == 5)

## 3. NVE Magasinstatistikk API: areas and time columns
The CSV from part 1 is gone; the data comes straight from the API at
`https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/`. It has four endpoints:
`HentOffentligData` (all weekly data), `HentOffentligDataSisteUke` (last week),
`HentOffentligDataMinMaxMedian` (min, max and median of the last 20 years) and
`HentOmråder` (the areas). None of them takes parameters, so the whole history is
always returned and any time interval has to be cut out afterwards.

In [ ]:
BASE = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk"

# Which areas does the API cover?
areas = requests.get(f"{BASE}/HentOmr%C3%A5der", timeout=60).json()[0]
rows = [
    {"group": group, **{k: a[k] for k in ("omrType", "omrnr", "navn", "beskrivelse")}}
    for group, items in areas.items() for a in items
]
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(rows)

**Areas covered:** `NO` is Norway as a whole (number 0), `EL` are the five electricity
price areas NO1-NO5 (Elspot areas), and `VASS` are watercourse areas 1-4. The area list
names four watercourse areas, but the data only contains VASS1-VASS3 (checked below).

In [ ]:
raw = requests.get(f"{BASE}/HentOffentligData", timeout=60).json()
df = pd.DataFrame(raw)
print(f"{df.shape[0]} rows x {df.shape[1]} columns")
df.head()

**Time columns** (my experience from part 1): the rows arrive **shuffled**, so the data
must be sorted before any line plot. `dato_Id` is a text date (the Sunday that ends the
week), `iso_aar`/`iso_uke` are the ISO year and week, and `neste_Publiseringsdato` holds
the placeholder `0001-01-01` for old rows. The code below converts the dates, sorts, and
checks that the ISO week and the date agree.

In [ ]:
COLUMN_NAMES = {
    "dato_Id": "date", "omrType": "area_type", "omrnr": "area_number",
    "iso_aar": "year", "iso_uke": "week", "fyllingsgrad": "fill_level",
    "kapasitet_TWh": "capacity_TWh", "fylling_TWh": "stored_TWh",
    "neste_Publiseringsdato": "next_publish_date",
    "fyllingsgrad_forrige_uke": "fill_level_prev_week",
    "endring_fyllingsgrad": "fill_level_change",
}
df = df.rename(columns=COLUMN_NAMES)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["area_type", "area_number", "date"]).reset_index(drop=True)
df = df.drop(columns="next_publish_date")      # publishing metadata only

df["area"] = [
    "Norway" if t == "NO" else (f"NO{n}" if t == "EL" else f"Watercourse {n}")
    for t, n in zip(df["area_type"], df["area_number"])
]

# Check the time columns: the ISO week of the date should equal the iso_uke column
iso = df["date"].dt.isocalendar()
print("ISO week matches:", (iso["week"].astype(int) == df["week"]).mean().round(4))
print("Weekday of the dates (6 = Sunday):", df["date"].dt.weekday.unique())
print("Weeks per area:")
df.groupby("area")["date"].agg(weeks="count", first="min", last="max")

## 4. Part 1 plots, recreated from the API
The same plots as in part 1 for the Norway total: each column on its own, then all
columns together as min-max scaled lines and as small multiples.

In [ ]:
MEASURES = ["fill_level", "capacity_TWh", "stored_TWh", "fill_level_prev_week", "fill_level_change"]
LABELS = {
    "fill_level": "Fill level (share of capacity)",
    "capacity_TWh": "Capacity (TWh)",
    "stored_TWh": "Stored energy (TWh)",
    "fill_level_prev_week": "Fill level previous week (share)",
    "fill_level_change": "Weekly change in fill level",
}
norway = df[df["area_type"] == "NO"].set_index("date")[MEASURES]

for i, col in enumerate(MEASURES):
    fig, ax = plt.subplots()
    ax.plot(norway.index, norway[col], color=PALETTE[i])
    ax.set_title(f"Norway - {LABELS[col]}, weekly (NVE API)")
    ax.set_xlabel("Date")
    ax.set_ylabel(LABELS[col])
    if col == "fill_level_change":
        ax.axhline(0, color="grey", linewidth=0.8)
    plt.show()

In [ ]:
# All columns together, last three years of the data
recent = norway.loc[norway.index.max() - pd.DateOffset(years=3):]

# Min-max scaling; constant columns (max == min) cannot be scaled and are left out
spread = recent.max() - recent.min()
constant = spread[spread == 0].index.tolist()
print("Constant, left out of the scaled plot:", constant)
scaled = (recent - recent.min()) / spread

fig, ax = plt.subplots(figsize=(11, 4.5))
for i, col in enumerate(MEASURES):
    if col in constant:
        continue
    ax.plot(scaled.index, scaled[col], color=PALETTE[i],
            linestyle="--" if col == "stored_TWh" else "-", label=LABELS[col])
ax.set_title("Norway - all columns scaled to 0-1 (min-max), last three years")
ax.set_xlabel("Date")
ax.set_ylabel("Scaled value (0 = min, 1 = max)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.show()

# Small multiples: one panel per column, own units, shared time axis
fig, axes = plt.subplots(len(MEASURES), 1, figsize=(11, 10), sharex=True)
for i, (ax, col) in enumerate(zip(axes, MEASURES)):
    ax.plot(recent.index, recent[col], color=PALETTE[i])
    ax.set_ylabel(LABELS[col].replace(" (", "\n("), fontsize=9)
axes[0].set_title("Norway - all columns in their own units, last three years")
axes[-1].set_xlabel("Date")
fig.tight_layout()
plt.show()

## 5. ENTSO-E: cross-border flows from/to Norway
The ENTSO-E Transparency Platform API needs a security token (account on
transparency.entsoe.eu, then an e-mail to ENTSO-E asking for *Restful API access*).
The token is read from `No_sync/ENTSOE`. I use the Python package `entsoe-py`, which
builds the requests and parses the XML answer.

A **connection** is one border between a Norwegian bidding zone and a neighbour
(for example `NO_2-DK_1`). Each connection has two **directions**: `out` (physical flow
from Norway to the neighbour) and `in` (from the neighbour into Norway). Values are
hourly power in MW, so one hourly value equals that many MWh.

Period: the last two years, up to the start of today (UTC).

In [ ]:
from entsoe import EntsoePandasClient

token = (SECRETS / "ENTSOE").read_text().strip()
client = EntsoePandasClient(api_key=token)

# Border list: Norwegian bidding zone first, neighbour second
CONNECTIONS = [
    ("NO_1", "SE_3"), ("NO_2", "DK_1"), ("NO_2", "NL"), ("NO_2", "DE_LU"),
    ("NO_2", "GB"), ("NO_3", "SE_2"), ("NO_4", "SE_1"), ("NO_4", "SE_2"), ("NO_4", "FI"),
]
END = pd.Timestamp.now(tz="UTC").normalize()
START = END - pd.DateOffset(years=2)
print(START, "->", END)

In [ ]:
import time
from entsoe.exceptions import NoMatchingDataError

CACHE = Path("../data/cache")          # git-ignored; avoids downloading twice
CACHE.mkdir(parents=True, exist_ok=True)


def hourly_flow(frm, to):
    """Physical flow frm -> to in MW as an hourly series (cached on disk)."""
    path = CACHE / f"flow_{frm}_{to}_{START:%Y%m%d}_{END:%Y%m%d}.csv"
    if path.exists():
        s = pd.read_csv(path, index_col=0).iloc[:, 0]
        s.index = pd.to_datetime(s.index, utc=True)
        return s
    s = client.query_crossborder_flows(frm, to, start=START, end=END)
    s = s.tz_convert("UTC").resample("1h").mean()      # some borders report 15 min values
    s.to_frame("mw").to_csv(path)
    return s


frames, skipped = [], []
for no_zone, other in CONNECTIONS:
    name = f"{no_zone}-{other}"
    for direction, (a, b) in {"out": (no_zone, other), "in": (other, no_zone)}.items():
        try:
            s = hourly_flow(a, b)
        except NoMatchingDataError:
            skipped.append((name, direction))
            continue
        frames.append(pd.DataFrame({"connection": name, "direction": direction,
                                    "ts": s.index, "mw": s.values}))
        time.sleep(0.3)        # be gentle with the API

flows = pd.concat(frames, ignore_index=True).dropna(subset=["mw"])
print("No data for:", skipped)
print(flows.groupby(["connection", "direction"]).size().unstack())
flows.head()

## 6. Spark: insert into Cassandra
Cassandra needs the table with its keys defined first. The **partition key**
`(connection, direction)` keeps one line of the plots together on one node, and the
**clustering column** `ts` sorts the rows in time order inside a partition.

In [ ]:
from cassandra.cluster import Cluster

session = Cluster(["127.0.0.1"]).connect()
session.execute("""
    CREATE TABLE IF NOT EXISTS ind320.crossborder (
        connection text, direction text, ts timestamp, mw double,
        PRIMARY KEY ((connection, direction), ts))
""")
session.execute("TRUNCATE ind320.crossborder")        # re-running the notebook starts clean
session.shutdown()

In [ ]:
from pyspark.sql import functions as F

# pandas -> Spark DataFrame. The timestamps go over as UTC *text* and are converted in
# Spark (session time zone = UTC). Handing pandas timestamps to Spark directly would
# reinterpret them in the Windows local time zone, which shifts them and merges the two
# identical hours when the clocks go back (Cassandra then overwrites one of them).
sdf = (spark.createDataFrame(flows.assign(ts=flows["ts"].dt.strftime("%Y-%m-%d %H:%M:%S")))
       .withColumn("ts", F.to_timestamp("ts")))
(sdf.write.format("org.apache.spark.sql.cassandra")
    .options(table="crossborder", keyspace="ind320")
    .mode("append").save())
print("rows written:", sdf.count())

## 7. Spark: extract from Cassandra
Read the table back with Spark and keep only what the plots need. Spark then
aggregates hourly values into one record per connection, direction and month: the
list of hourly values (for the line plot) and their sum (for the pie chart).

In [ ]:
from pyspark.sql import functions as F

cb = (spark.read.format("org.apache.spark.sql.cassandra")
      .options(table="crossborder", keyspace="ind320").load()
      .select("connection", "direction", "ts", "mw"))
print("rows read back from Cassandra:", cb.count())

monthly = (
    cb.withColumn("year", F.year("ts")).withColumn("month", F.month("ts"))
      .groupBy("connection", "direction", "year", "month")
      .agg(F.sort_array(F.collect_list(F.struct("ts", "mw"))).alias("hourly"),
           F.round(F.sum("mw"), 1).alias("total_mwh"),
           F.count("mw").alias("hours"),
           F.date_format(F.min("ts"), "yyyy-MM-dd'T'HH:mm:ss").alias("start"))   # UTC text
      .orderBy("connection", "direction", "year", "month")
)
monthly.select("connection", "direction", "year", "month", "hours", "total_mwh").show(8)

## 8. Plots: pie chart and first-month line plot
**Pie chart:** total energy for one year on one chosen connection, split into the part
that flowed out of Norway and the part that flowed in. **Line plot:** the first month
of the chosen year, one line per connection and direction.

In [ ]:
# Small pandas copy of the Spark result for plotting (monthly rows, not hourly)
m = monthly.select("connection", "direction", "year", "month", "total_mwh").toPandas()

# Choose the latest year that has all 12 months; the connection is chosen by hand
full_years = m.groupby("year")["month"].nunique()
YEAR = int(full_years[full_years == 12].index.max())
CONNECTION = "NO_2-DK_1" if "NO_2-DK_1" in set(m["connection"]) else m["connection"].iloc[0]
print("chosen year:", YEAR, "| chosen connection:", CONNECTION)

# Pie chart: yearly totals in GWh, out of vs into Norway
year = m[(m["year"] == YEAR) & (m["connection"] == CONNECTION)].groupby("direction")["total_mwh"].sum() / 1000
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(year.values, labels=[f"{d} ({v:,.0f} GWh)" for d, v in year.items()],
       colors=PALETTE[:len(year)], autopct="%1.1f%%", startangle=90,
       wedgeprops={"edgecolor": "white"})
ax.set_title(f"{CONNECTION}, {YEAR}: energy out of vs into Norway")
ax.grid(False)
plt.show()

In [ ]:
# Line plot: first month (January) of the chosen year, one line per connection+direction
jan = (monthly.filter((F.col("year") == YEAR) & (F.col("month") == 1))
       .select("connection", "direction", F.explode("hourly").alias("h"))
       .select("connection", "direction", F.col("h.ts").alias("ts"), F.col("h.mw").alias("mw"))
       .toPandas())

fig, ax = plt.subplots(figsize=(11, 5))
for i, ((conn, d), g) in enumerate(jan.groupby(["connection", "direction"])):
    ax.plot(g["ts"], g["mw"], label=f"{conn} {d}", color=PALETTE[i % len(PALETTE)],
            linestyle="-" if d == "out" else "--", linewidth=1)
ax.set_title(f"Hourly power flow, January {YEAR} (out = Norway to neighbour, in = neighbour to Norway)")
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Power (MW)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8)
plt.show()

## 9. Spark data into MongoDB
The Spark-extracted monthly records go to MongoDB. One document per connection,
direction and month holds the list of hourly values, which is compact (a few hundred
documents instead of hundreds of thousands) and exactly what the app needs: sum the
`total_mwh` for the pie chart, or plot `values` for the line chart.

In [ ]:
docs = []
for r in monthly.collect():                       # a few hundred rows
    docs.append({
        "connection": r["connection"], "direction": r["direction"],
        "year": r["year"], "month": r["month"], "hours": r["hours"],
        "total_mwh": r["total_mwh"],
        "start": r["start"],                                 # first hour of the month (UTC)
        "values": [round(h["mw"], 1) for h in r["hourly"]],  # hourly MW
    })

col = mongo["ind320"]["crossborder_monthly"]
col.delete_many({})                               # re-running replaces the old copy
col.insert_many(docs)
col.create_index([("connection", 1), ("direction", 1), ("year", 1), ("month", 1)])
print("documents in MongoDB:", col.count_documents({}))
print("example:", {k: (v if k != "values" else f"[{len(v)} values]") for k, v in docs[0].items()})

## 10. AI usage
AI_USAGE_TEXT

## 11. Log
LOG_TEXT